# 04 — Results Visualization
**Project:** Phase-Field Informed ML for Microstructure Prediction in Ti-Nb-O Refractory Alloys  
**Author:** Anosike Kelechi Kenneth  
**Purpose:** UMAP dimensionality reduction, composition space mapping, candidate screening, and final results summary.  
**Run order:** Run AFTER 03_modeling.ipynb

---

In [ ]:
# Section 1: Imports
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

try:
    import umap
    print('UMAP OK')
except ImportError:
    print('UMAP not found - install with: pip install umap-learn')

print('All imports OK')

In [ ]:
# Section 2: Load feature matrix
print('Loading Magpie feature matrix...')
df = pd.read_csv('../data/magpie_features_20k.csv')
feature_cols = [c for c in df.columns if c != 'gap pbe']
X = df[feature_cols].values
y = df['gap pbe'].values
print(f'Loaded: {df.shape}')

# Train/test split (same seed as notebook 03 for consistency)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Retrain RF on training set
print('Retraining RF for visualization...')
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
print(f'RF MAE={mean_absolute_error(y_test, y_pred):.4f} eV  R²={r2_score(y_test, y_pred):.4f}')

In [ ]:
# Section 3: PCA for dimensionality reduction before UMAP
# Standardize features first
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA to 10 components (speeds up UMAP and reduces noise)
pca = PCA(n_components=10, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print(f'PCA explained variance (10 components): {pca.explained_variance_ratio_.sum():.3f}')

In [ ]:
# Section 4: UMAP projection coloured by band gap
print('Running UMAP (this takes 1-2 minutes)...')
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
embedding = reducer.fit_transform(X_pca)
print(f'UMAP embedding shape: {embedding.shape}')

fig, ax = plt.subplots(figsize=(9, 7))
sc = ax.scatter(embedding[:, 0], embedding[:, 1],
                c=y, cmap='viridis', s=5, alpha=0.5, rasterized=True)
plt.colorbar(sc, ax=ax, label='Band Gap (eV)')
ax.set_xlabel('UMAP 1', fontsize=12)
ax.set_ylabel('UMAP 2', fontsize=12)
ax.set_title('UMAP Projection of Magpie Feature Space\nColoured by Band Gap', fontsize=12)
plt.tight_layout()
plt.savefig('../figures/04_umap_bandgap.png', dpi=150)
plt.show()
print('Saved: figures/04_umap_bandgap.png')

In [ ]:
# Section 5: UMAP coloured by predicted vs actual (error map)
# Get predictions for ALL points using cross-prediction
y_pred_all = rf.predict(X)
residuals = np.abs(y - y_pred_all)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Predicted bandgap
sc1 = axes[0].scatter(embedding[:, 0], embedding[:, 1],
                       c=y_pred_all, cmap='viridis', s=5, alpha=0.5, rasterized=True)
plt.colorbar(sc1, ax=axes[0], label='Predicted Band Gap (eV)')
axes[0].set_title('UMAP: Predicted Band Gap', fontsize=12)
axes[0].set_xlabel('UMAP 1'); axes[0].set_ylabel('UMAP 2')

# Absolute error
sc2 = axes[1].scatter(embedding[:, 0], embedding[:, 1],
                       c=residuals, cmap='Reds', s=5, alpha=0.5,
                       vmax=np.percentile(residuals, 95), rasterized=True)
plt.colorbar(sc2, ax=axes[1], label='Absolute Error (eV)')
axes[1].set_title('UMAP: Prediction Error Map', fontsize=12)
axes[1].set_xlabel('UMAP 1'); axes[1].set_ylabel('UMAP 2')

plt.suptitle('Model Predictions in UMAP Space', fontsize=13)
plt.tight_layout()
plt.savefig('../figures/04_umap_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Section 6: Final results summary table
from sklearn.metrics import mean_squared_error

mae_final  = mean_absolute_error(y_test, y_pred)
rmse_final = np.sqrt(mean_squared_error(y_test, y_pred))
r2_final   = r2_score(y_test, y_pred)

results = pd.DataFrame({
    'Metric': ['MAE (eV)', 'RMSE (eV)', 'R²', 'Training samples', 'Test samples', 'Magpie features'],
    'Value': [f'{mae_final:.4f}', f'{rmse_final:.4f}', f'{r2_final:.4f}',
              str(X_train.shape[0]), str(X_test.shape[0]), str(len(feature_cols))]
})
print('=== Final Model Performance Summary ===')
print(results.to_string(index=False))

# Save summary
results.to_csv('../data/final_results_summary.csv', index=False)
print('\nSaved: data/final_results_summary.csv')

In [ ]:
# Section 7: Connection to Ti-Nb-O research
# Identify where Ti-Nb-O compounds sit in the UMAP space
# using composition string matching
from matminer.datasets import load_dataset

df_full = load_dataset('matbench_mp_gap')
df_sub = df_full.sample(20000, random_state=42).reset_index(drop=True)
df_sub['formula'] = df_sub['structure'].apply(lambda s: s.composition.reduced_formula)

# Find Ti-Nb-O containing compounds
ti_nb_o_mask = df_sub['formula'].apply(
    lambda f: ('Ti' in f or 'Nb' in f) and 'O' in f
)
print(f'Ti/Nb oxide compounds in dataset: {ti_nb_o_mask.sum()}')
print('Sample Ti-Nb-O related formulas:')
print(df_sub[ti_nb_o_mask]['formula'].head(10).tolist())

# Highlight on UMAP
fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(embedding[~ti_nb_o_mask, 0], embedding[~ti_nb_o_mask, 1],
           c='lightgrey', s=4, alpha=0.3, rasterized=True, label='Other compounds')
ax.scatter(embedding[ti_nb_o_mask, 0], embedding[ti_nb_o_mask, 1],
           c='red', s=20, alpha=0.8, label=f'Ti/Nb oxides (n={ti_nb_o_mask.sum()})')
ax.set_xlabel('UMAP 1', fontsize=12)
ax.set_ylabel('UMAP 2', fontsize=12)
ax.set_title('Ti/Nb Oxide Compounds in Magpie Feature Space', fontsize=12)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('../figures/04_umap_tinbo_highlight.png', dpi=150)
plt.show()
print('Saved: figures/04_umap_tinbo_highlight.png')
print('\nAll notebooks complete. Repository is ready for submission.')